# Serving your model

A model on disk isn't a product. This notebook puts your fine-tuned router from module 07 behind an API, wraps it in a backend service the way a real app's backend would call it, and measures what it costs to run. You'll finish with your own numbers for latency, throughput, and the traffic level at which self-hosting beats a hosted API.

The stack you're building:

```
student question
   -> FastAPI service (app.py): validation, retries, fallback, timing
   -> OpenAI-compatible model server: mlx_lm.server on your Mac (or LM Studio, Ollama, vLLM)
   -> your fine-tuned model
```

Every model server in this notebook speaks the same OpenAI-style API you used in module 01, so the backend doesn't care which one is underneath. `app.py` reads the server's address from the `MODEL_BASE_URL` environment variable (with `MODEL_NAME` and `MODEL_API_KEY` alongside it). You can develop against your Mac and deploy against a rented GPU or a hosted endpoint by changing that one variable, with no code changes.

## In this notebook

1. Latency: p50 and p95 for single requests.
2. Concurrency: what happens to throughput and latency with 1, 4, and 8 requests in flight.
3. The backend service: `app.py`, with validation, a retry, and a safe fallback.
4. What does it cost? Hosted API vs. rented GPU vs. your Mac.
5. Other ways to serve the same model: LM Studio, Ollama, and vLLM.

## Before you start

1. Finish module 07. You need `lessons/07_mlx_lora/models/darvis-router-1.5b-4bit`. Section 5's Ollama and vLLM routes also use the 16-bit fused model next to it, `models/darvis-router-1.5b`.
2. Run the install cell below once.
3. In an iTerm2 tab, from `ML_Lab`, start the model server and leave it running for the whole notebook:

   ```bash
   uv run mlx_lm.server --model lessons/07_mlx_lora/models/darvis-router-1.5b-4bit --port 8080
   ```

   It's an OpenAI-compatible server built into `mlx-lm`, the quickest way to serve an MLX model. Give it a few seconds to load the model before you run the setup cell, which sends it a first request.
4. Section 3 needs a second iTerm2 tab for the FastAPI service. The notebook tells you when.
5. No downloads. The kernel is the `.venv` in `ML_Lab`.

In [ ]:
# One-time step: adds FastAPI and uvicorn to ML_Lab's pyproject.toml and uv.lock.
# Safe to rerun. uv finds the ML_Lab project from this folder. Commit both files afterward.
!uv add fastapi uvicorn

In [ ]:
import asyncio
import json
import os
import re
import statistics
import time

import httpx
import numpy as np
from openai import AsyncOpenAI, OpenAI

MODEL_URL = "http://localhost:8080/v1"
MODEL = "default_model"     # mlx_lm.server maps this name to the model it was started with
SYSTEM = "You are the Darvis query router. Convert the student's question into a JSON query."

client = OpenAI(base_url=MODEL_URL, api_key="local")
aclient = AsyncOpenAI(base_url=MODEL_URL, api_key="local")

def ask(question, temperature=0.0):
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}],
        temperature=temperature,
        max_tokens=100,
    )
    return r

r = ask("who's easier for data structures, alvarez or chen? fall '26")
print(r.choices[0].message.content)
print(r.usage)

That's the same `openai` client and the same code shape as module 01. Only the URL changed. Your fine-tuned model is now a drop-in replacement for any OpenAI-compatible model in any app.

## 1. Latency

Users feel latency, and averages hide the slow requests. Measure percentiles: p50 is the typical request, and p95 is the experience of your unluckiest 1 in 20 users.

In [ ]:
QUESTIONS = [
    "who's easier for data structures, alvarez or chen? fall '26",
    "prereqs for cs3214?",
    "what is lin alg about",
    "best professor for comp org 2 next spring",
    "is okafor teaching systems this fall",
    "avg gpa MATH 2204",
    "compare rossi and haddad in algorithms",
    "where can i print on campus",
    "how much work is the ml class",
    "can i take cs 3114 without CS 2114",
    "who teaches dsa best",
    "is CS 4104 with Dr. Lindqvist a gpa booster",
    "when is mobile dev usually offered",
    "Nakamura or Duarte for stats for engineers?",
    "what topics does comp sys cover",
    "how do i reset my vt password",
]

latencies, out_tokens = [], []
for q in QUESTIONS:
    start = time.perf_counter()
    r = ask(q)
    latencies.append(time.perf_counter() - start)
    out_tokens.append(r.usage.completion_tokens if r.usage else len(r.choices[0].message.content) / 4)

lat_ms = np.array(latencies) * 1000
print(f"requests: {len(lat_ms)}")
print(f"p50 {np.percentile(lat_ms, 50):.0f} ms   p95 {np.percentile(lat_ms, 95):.0f} ms   max {lat_ms.max():.0f} ms")
print(f"average output: {np.mean(out_tokens):.0f} tokens   sequential throughput: {len(lat_ms) / sum(latencies):.2f} requests/s")

## 2. Concurrency

Real traffic arrives in parallel. Send the same questions with 1, 4, and 8 requests in flight at once and watch what happens to throughput and to per-request latency.

In [ ]:
async def timed_request(question, semaphore):
    async with semaphore:
        start = time.perf_counter()
        await aclient.chat.completions.create(
            model=MODEL,
            messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}],
            temperature=0.0,
            max_tokens=100,
        )
        return time.perf_counter() - start

async def load_test(concurrency, n_requests=32):
    semaphore = asyncio.Semaphore(concurrency)
    questions = [QUESTIONS[i % len(QUESTIONS)] for i in range(n_requests)]
    start = time.perf_counter()
    times = await asyncio.gather(*(timed_request(q, semaphore) for q in questions))
    wall = time.perf_counter() - start
    ms = np.array(times) * 1000
    print(f"concurrency {concurrency:2}: {n_requests / wall:5.2f} req/s   p50 {np.percentile(ms, 50):6.0f} ms   p95 {np.percentile(ms, 95):6.0f} ms")

for c in [1, 4, 8]:
    await load_test(c)

How to read this. If throughput rises with concurrency, the server is batching: it runs several requests through the model together, which uses the hardware better. If throughput stays flat while latency climbs, requests are just queueing. `mlx_lm.server` batches to a degree on Apple Silicon, while production servers like vLLM are built around continuous batching and scale much further on a GPU.

The number to carry forward is your peak requests per second. It's the capacity of one Mac (or one GPU) serving this model.

## 3. The backend service

A raw model endpoint isn't safe to expose to users. It can return malformed JSON, invent intents, or hang. `app.py` in this folder wraps it with:

- Input validation (Pydantic rejects empty or huge questions before they reach the model)
- Output validation against the exact schema, including the allowed intents and terms
- One retry on bad output, then a safe fallback instead of an error
- Timing on every request

Start it now:

1. Open a second iTerm2 tab (leave `mlx_lm.server` running in the first) and, from `ML_Lab`, run:

   ```bash
   cd lessons/08_serving
   uv run uvicorn app:app --port 8000 --reload
   ```

   `app.py` points at `http://localhost:8080/v1` by default, which is the server from "Before you start". `--reload` restarts the service whenever you save `app.py`, which you'll use in the exercises.
2. Open http://localhost:8000/docs in your browser. FastAPI generates interactive API docs from the code, so you can try the `/route` endpoint there: expand `POST /route`, click "Try it out", type a question, and click "Execute".
3. Come back and run the cells below, which call the same endpoints from Python.

In [ ]:
API = "http://localhost:8000"

print(httpx.get(f"{API}/health", timeout=10).json())
print()
for q in QUESTIONS[:6]:
    r = httpx.post(f"{API}/route", json={"question": q}, timeout=30).json()
    print(f"{q!r}\n  -> {r['route']}  attempts={r['attempts']} fallback={r['fallback']} {r['latency_ms']:.0f} ms")

In [ ]:
# Bad input never reaches the model. FastAPI answers with a 422 and an explanation.
r = httpx.post(f"{API}/route", json={"question": ""}, timeout=30)
print(r.status_code)
print(json.dumps(r.json(), indent=2)[:400])

## 4. What does it cost?

Three options for running the router:

1. A hosted API (OpenAI, Anthropic, Together, and others): you pay per token, nothing when idle, and it scales automatically.
2. A rented GPU running vLLM: a fixed hourly price whether you use it or not, with capacity set by throughput.
3. Your Mac: effectively free, but it isn't production. It sleeps, it's on home internet, and it's your laptop.

The prices below are placeholders. Look up current prices for a hosted model you'd actually consider and for a GPU you'd rent, and replace them. The structure of the answer matters more than these numbers.

In [ ]:
# Measured from your model (section 1). Input tokens include the chat template around the short system prompt.
avg_in_tokens = 60
avg_out_tokens = float(np.mean(out_tokens))

# Placeholders: replace with real prices
HOSTED_IN_PRICE = 0.50     # $ per 1M input tokens
HOSTED_OUT_PRICE = 1.50    # $ per 1M output tokens
GPU_HOURLY = 0.60          # $ per hour for a rented GPU
GPU_REQ_PER_SEC = 20       # replace with a measured vLLM number; it's typically far above your Mac's

def hosted_monthly(requests):
    return requests * (avg_in_tokens * HOSTED_IN_PRICE + avg_out_tokens * HOSTED_OUT_PRICE) / 1e6

gpu_monthly = GPU_HOURLY * 730   # always-on
gpu_capacity = GPU_REQ_PER_SEC * 3600 * 730

for requests in [10_000, 100_000, 1_000_000, 10_000_000]:
    print(f"{requests:>12,} req/month   hosted ${hosted_monthly(requests):>10,.2f}   one GPU ${gpu_monthly:>8,.2f}"
          + ("" if requests <= gpu_capacity else "  (over one GPU's capacity)"))

per_request = hosted_monthly(1)
print(f"\nbreak-even: about {gpu_monthly / per_request:,.0f} requests/month")

Notice what this shows. Because your fine-tuned model needs only a one-line prompt and a 1.5B model is cheap per token, hosted per-request cost is tiny, so a dedicated GPU only wins at high volume. That's typical: self-hosting usually wins on control, privacy, latency, and owning the model, and only wins on price at scale. The fine-tuning itself is what made the per-request cost small, whichever way you host it.

For a small app like Darvis, a reasonable plan: the router runs on whatever is cheapest to operate at current traffic (a hosted endpoint serving your fine-tuned model, or a small GPU instance), behind the same FastAPI contract, so you can switch later by changing `MODEL_BASE_URL`.

## 5. Other ways to serve the same model

`mlx_lm.server` is the fastest path on a Mac, but it's one option of several. Because `app.py` only talks to an OpenAI-compatible URL, each option below plugs in by setting `MODEL_BASE_URL` and `MODEL_NAME` when you start uvicorn. None of these are required to finish the module. When you switch, stop the old uvicorn (Ctrl+C) so port 8000 is free, and stop `mlx_lm.server` too if you want its memory back.

### LM Studio

LM Studio can load MLX models from its models folder, which expects a `publisher/model-name` layout.

1. Find the models folder in LM Studio's My Models view.
2. Copy your model in, for example to `~/.lmstudio/models/pujanpatel8/darvis-router-1.5b-4bit`. If your models folder is the default `~/.lmstudio/models`, from `ML_Lab`:

   ```bash
   mkdir -p ~/.lmstudio/models/pujanpatel8
   cp -R lessons/07_mlx_lora/models/darvis-router-1.5b-4bit ~/.lmstudio/models/pujanpatel8/
   ```

3. Run `lms ls` to confirm LM Studio sees it.
4. Load it and start the server on port 1234, as in module 01.
5. Start the FastAPI app pointed at LM Studio, from `lessons/08_serving`:

   ```bash
   MODEL_BASE_URL=http://localhost:1234/v1 MODEL_NAME=<the id lms ls shows> uv run uvicorn app:app --port 8000 --reload
   ```

### Ollama (optional, more steps)

Ollama runs GGUF files, the single-file format from llama.cpp (module 01 compared GGUF and MLX). So you convert first, starting from the 16-bit fused model: the converter reads ordinary Hugging Face-format weights, not MLX's packed 4-bit format, and quantizing from 16-bit loses less than quantizing twice. `Q4_K_M` is one of llama.cpp's 4-bit quantization schemes and a common default. The commands below reflect llama.cpp's current tooling. If a script name has changed, check the llama.cpp GitHub repo.

1. Convert to a 16-bit GGUF, then quantize it to 4-bit:

   ```bash
   brew install llama.cpp                                    # gives you llama-quantize
   git clone --depth 1 https://github.com/ggml-org/llama.cpp ~/Developer/llama.cpp
   cd ~/Developer/ML_Lab/lessons/07_mlx_lora
   uv run --with-requirements ~/Developer/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt \
     python ~/Developer/llama.cpp/convert_hf_to_gguf.py models/darvis-router-1.5b \
     --outfile models/darvis-router-f16.gguf --outtype f16
   llama-quantize models/darvis-router-f16.gguf models/darvis-router-q4_k_m.gguf Q4_K_M
   ```

2. Make a file called `Modelfile` next to the GGUF (in `lessons/07_mlx_lora/models/`, which git ignores):

   ```
   FROM ./darvis-router-q4_k_m.gguf
   PARAMETER temperature 0
   SYSTEM You are the Darvis query router. Convert the student's question into a JSON query.
   ```

3. Register the model with Ollama and try it:

   ```bash
   cd ~/Developer/ML_Lab/lessons/07_mlx_lora/models
   ollama create darvis-router -f Modelfile
   ollama run darvis-router "prereqs for systems?"
   ```

4. Ollama serves it at `http://localhost:11434/v1`. Start the FastAPI app with `MODEL_BASE_URL=http://localhost:11434/v1 MODEL_NAME=darvis-router`.

### vLLM on a rented GPU (production)

vLLM is the standard open-source server for production traffic. Two ideas make it fast:

- Continuous batching. A naive server waits for a whole batch of requests to finish before starting the next. vLLM adds new requests to the running batch at every generation step, as soon as any sequence finishes, so the GPU stays busy.
- PagedAttention. Every request needs a KV cache (the stored keys and values from module 06's attention). Instead of reserving one big block per request sized for the longest possible output, vLLM hands out small fixed-size pages on demand, the way an operating system pages memory. Less memory is wasted, so more requests fit on the GPU at once.

Together, they let one GPU serve many users at once. To run it:

1. Rent a Linux GPU box (RunPod, Lambda, and others). A 1.5B model fits on almost any current GPU.
2. Copy up the 16-bit fused model, `lessons/07_mlx_lora/models/darvis-router-1.5b` (with `scp`, or push it to a private Hub repo as in module 07 and download it on the box).
3. On the box, run:

   ```bash
   uv venv && uv pip install vllm
   uv run vllm serve ./darvis-router-1.5b --served-model-name darvis-router --port 8000
   ```

   This is the one place the course uses `uv pip`. It's a throwaway environment on a rented machine, not the `ML_Lab` project.
4. Point the FastAPI service at it with `MODEL_BASE_URL=http://<box address>:8000/v1 MODEL_NAME=darvis-router`.

Put it behind authentication before exposing it to the internet. An open endpoint lets anyone who finds it run your GPU on your bill. The simplest start is vLLM's `--api-key` flag, with `MODEL_API_KEY` set to the same value when you start the FastAPI service.

## Wrap-up

Why this matters. This module closes the loop from weights to product: train a specialized model, serve it, put a reliable backend in front of it, and know the cost per request. The backend patterns here (schema validation, a retry, a safe fallback, timing every request) are what make an unreliable component like an LLM safe to put in front of users in any product. The cost math tells you whether self-hosting is worth it at your traffic level, and that answer changes as you grow. Keeping the backend contract fixed while the model server behind it can move is what lets you act on that answer cheaply.

You're done when you can answer, with your own measured numbers: what's the p95 latency of the router, how many requests per second can your Mac handle, and at what monthly request volume a rented GPU beats a hosted API.

Going further: the vLLM documentation covers deployment, quantized models, and multi-GPU serving. The PagedAttention paper (Kwon et al., 2023, "Efficient Memory Management for Large Language Model Serving with PagedAttention") explains the memory trick from section 5 in depth.

---
# Exercises

**Exercise 1: Caching.** Students ask the same questions constantly. Add an in-memory cache to `app.py`: normalize the question (lowercase, collapse whitespace), and if it's been seen, return the stored route without calling the model. Add a `cached: bool` field to the response. Measure latency for a repeated question before and after.

**Exercise 2: Logging for future training data.** Make `app.py` append one JSON line per request to `logs/requests.jsonl` with a timestamp, the question, the route, attempts, fallback, and latency. This file is the seed of your next, better training set (module 07, exercise 2). Before doing this with real users, decide what you'll tell them and how long you'll keep it. Questions can contain personal information. Add `logs/` to `.gitignore`.

**Exercise 3: Load test the service.** Using `httpx.AsyncClient`, send 64 requests to `/route` with 8 in flight at a time. Report requests per second, p50 and p95 latency, and the fallback rate. Compare with the raw model numbers from section 2. How much overhead does the FastAPI layer add?

**Exercise 4: Fine-tuned vs. hosted.** If you have an API key for a hosted model, score it on module 07's test set (`../07_mlx_lora/data/test.jsonl`) using the detailed system prompt from module 07, and compare exact-match accuracy, p50 latency, and estimated cost per 1,000 requests with your fine-tuned model.

**Exercise 5: Deployment plan.** Write half a page, for yourself, on how the router would ship inside an app like Darvis: where the model runs, how the FastAPI router is called from the app's existing backend, what happens when the model server is down, how you'll monitor accuracy after launch, and the monthly cost at a realistic traffic level. This is the kind of document you'll write for every infrastructure decision as a founder.

---
# Solutions

Exercises 1 and 2 are edits to `app.py`. Here's the changed part of the `/route` endpoint, with both a cache and logging. Replace the `route` function in `app.py` with this, add `cached: bool = False` to `RouteResponse`, and add `import datetime` at the top:

```python
CACHE: dict[str, Route] = {}
os.makedirs("logs", exist_ok=True)

def normalize(q: str) -> str:
    return re.sub(r"\s+", " ", q.strip().lower())

def log_request(question, result, attempts, fallback, latency_ms, cached):
    with open("logs/requests.jsonl", "a") as f:
        f.write(json.dumps({
            "ts": datetime.datetime.now(datetime.timezone.utc).isoformat(),
            "question": question, "route": result.model_dump(), "attempts": attempts,
            "fallback": fallback, "latency_ms": round(latency_ms, 1), "cached": cached,
        }) + "\n")

@app.post("/route", response_model=RouteResponse)
async def route(body: Question):
    start = time.perf_counter()
    key = normalize(body.question)
    if key in CACHE:
        latency = (time.perf_counter() - start) * 1000
        log_request(body.question, CACHE[key], 0, False, latency, True)
        return RouteResponse(route=CACHE[key], attempts=0, fallback=False, latency_ms=latency, cached=True)

    for attempt in range(1, MAX_ATTEMPTS + 1):
        response = await client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": body.question}],
            temperature=0.0, max_tokens=100,
        )
        parsed = extract_json(response.choices[0].message.content or "")
        if parsed is not None:
            try:
                result = Route.model_validate(parsed)
                CACHE[key] = result
                latency = (time.perf_counter() - start) * 1000
                log_request(body.question, result, attempt, False, latency, False)
                return RouteResponse(route=result, attempts=attempt, fallback=False, latency_ms=latency)
            except ValidationError:
                pass
    latency = (time.perf_counter() - start) * 1000
    log_request(body.question, FALLBACK, MAX_ATTEMPTS, True, latency, False)
    return RouteResponse(route=FALLBACK, attempts=MAX_ATTEMPTS, fallback=True, latency_ms=latency)
```

A production cache would live in Redis with an expiry so it's shared across server processes and survives restarts. The test below works once your edited `app.py` has reloaded.

In [ ]:
# Exercise 1 check
q = {"question": "prereqs for cs3214?"}
for i in range(3):
    r = httpx.post(f"{API}/route", json=q, timeout=30).json()
    print(f"call {i + 1}: {r['latency_ms']:7.1f} ms  cached={r.get('cached')}")

In [ ]:
# Exercise 3
async def load_test_api(n_requests=64, concurrency=8):
    semaphore = asyncio.Semaphore(concurrency)
    async with httpx.AsyncClient(timeout=60) as http:
        async def one(i):
            async with semaphore:
                start = time.perf_counter()
                r = await http.post(f"{API}/route", json={"question": QUESTIONS[i % len(QUESTIONS)] + f" ({i})"})
                return time.perf_counter() - start, r.json()["fallback"]
        start = time.perf_counter()
        results = await asyncio.gather(*(one(i) for i in range(n_requests)))
        wall = time.perf_counter() - start
    ms = np.array([t for t, _ in results]) * 1000
    fallbacks = sum(f for _, f in results)
    print(f"{n_requests / wall:.2f} req/s   p50 {np.percentile(ms, 50):.0f} ms   p95 {np.percentile(ms, 95):.0f} ms   fallback rate {fallbacks / n_requests:.0%}")

# The " (i)" suffix makes every question unique so a cache from exercise 1 doesn't inflate the numbers.
await load_test_api()

In [ ]:
# Exercise 4
HOSTED_URL = os.getenv("HOSTED_BASE_URL")        # for example https://api.openai.com/v1
HOSTED_KEY = os.getenv("HOSTED_API_KEY")
HOSTED_MODEL = os.getenv("HOSTED_MODEL")         # a small, cheap model is the fair comparison

def extract_json(text):
    m = re.search(r"\{.*\}", text or "", re.DOTALL)
    try:
        return json.loads(m.group(0)) if m else None
    except json.JSONDecodeError:
        return None

def exact_match(pred, target):
    if not isinstance(pred, dict):
        return False
    norm = lambda xs: sorted(str(x).strip().lower() for x in (xs or []))
    return (pred.get("intent") == target["intent"] and norm(pred.get("courses")) == norm(target["courses"])
            and norm(pred.get("professors")) == norm(target["professors"]) and pred.get("term") == target["term"])

with open("../07_mlx_lora/data/test.jsonl") as f:
    test_rows = [json.loads(line)["messages"] for line in f]

def evaluate(c, model, system, rows):
    hits, lat = 0, []
    for msgs in rows:
        start = time.perf_counter()
        r = c.chat.completions.create(model=model, temperature=0, max_tokens=100,
                                      messages=[{"role": "system", "content": system}, msgs[1]])
        lat.append(time.perf_counter() - start)
        hits += exact_match(extract_json(r.choices[0].message.content), json.loads(msgs[2]["content"]))
    return hits / len(rows), np.percentile(np.array(lat) * 1000, 50)

acc, p50 = evaluate(client, MODEL, SYSTEM, test_rows[:50])
print(f"fine-tuned (local): exact {acc:.0%}   p50 {p50:.0f} ms")

if HOSTED_URL and HOSTED_KEY and HOSTED_MODEL:
    DETAILED = "PASTE SYSTEM_DETAILED FROM MODULE 07 HERE"
    hosted = OpenAI(base_url=HOSTED_URL, api_key=HOSTED_KEY)
    acc_h, p50_h = evaluate(hosted, HOSTED_MODEL, DETAILED, test_rows[:50])
    print(f"hosted:             exact {acc_h:.0%}   p50 {p50_h:.0f} ms")
else:
    print("Set HOSTED_BASE_URL, HOSTED_API_KEY and HOSTED_MODEL before starting Jupyter to run the hosted comparison.")

Exercise 5 has no code. Write it in `experiments/` or Notion, and revisit it once you have real traffic numbers for the app.